# AMEX Enterprise Credit Risk Platform
## Notebook 43 -- Early Warning System: Modeling
### Phase 3 . Problem Statement 7: Early Warning System

CRISP-DM stage: **Modeling**. Sprint 1, Notebook 2 of 4 for this problem. Depends on Problem 1 Notebooks 01/02/05 (config, train/validation split membership, real champion AUC for reference) and this problem's own Notebook 42 (`early_warning_policy.json`).

**What this notebook does (real, computed on your machine when you run it):**
- Streams the real raw Kaggle `train_data.csv`, sorts each customer's statements chronologically by `S_2`, and for every monitored feature (Notebook 42's reused, correlation-filtered base D_* list) computes a real per-customer BASELINE mean and sample standard deviation from all statements EXCEPT the latest, plus the latest statement's raw value -- no model is trained; this is pure per-customer descriptive statistics
- Converts the latest value into a real z-score against that customer's own baseline for every monitored feature, honestly excluding a feature from a customer's score when a real z-score isn't computable (fewer than 2 baseline points, a zero-variance baseline, or a null value) rather than treating it as either deviating or not
- Computes the real `EARLY_WARNING_SCORE` (count of features whose latest z-score clears `Z_THRESHOLD`) for every customer with enough statement history, evaluated on the SAME validation-split population Problems 1/5/6 evaluated their holdout AUC on (reused purely for comparability -- there is no train/holdout leakage risk in an unsupervised, label-free computation)
- Reports the SECONDARY, non-gating ROC-AUC / PR-AUC / log-loss of the continuous (normalized) `EARLY_WARNING_SCORE` against the real eventual-default label, honestly disclosing that a lower AUC than Problems 1/5/6's trained classifiers is the expected outcome for a rule-based control-chart technique, not a failure
- Sweeps Notebook 42's real `MIN_DEVIATION_COUNT_CANDIDATES`, and at each candidate: computes the real default-rate LIFT among alerted vs. base-population customers (the PRIMARY KPI), checks it against the >=1.5x target, and reports the full classification metrics suite (Accuracy, Precision, Recall, F1, Specificity, MCC, confusion matrix) treating ALERT as the binary prediction -- the standing metrics-suite rule applied to this problem's own threshold-sweep structure
- Renders and saves a ROC curve, a Precision-Recall curve, and a default-rate-lift-by-candidate bar chart (reused directly by Notebooks 44/45's reports, not regenerated)
- Writes `early_warning_modeling_results.json` for Notebook 44 to consume

**What this notebook does NOT do:** it does not select a final winning candidate (that honest selection -- meets-KPI-or-best-flagged-NOT-RECOMMENDED -- is Notebook 44's job, the same separation of concerns Notebooks 39/40 and 35/36 established) and it trains no model at all (this is a genuinely unsupervised, rule-based technique).

Zero-fabrication: every number this notebook prints is computed live from your real Kaggle data on this run. `Z_THRESHOLD`, `MIN_STATEMENTS_FOR_BASELINE`, and the candidate sweep are reused verbatim from Notebook 42's policy, never re-guessed here.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
#             AND PROBLEM 6/7'S REAL OUTPUTS (NOTEBOOKS 38, 40, 42)
# =============================================================================
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 38, 40, 42")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"
NB42_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_42_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first"),
    (NB42_SUMMARY_PATH, "run 42_early_warning_system_business_understanding.ipynb first "
                         "(this notebook consumes its early_warning_policy.json)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)
with open(NB42_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB42_SUMMARY = json.load(f)

EARLY_WARNING_POLICY_PATH = Path(NB42_SUMMARY["policy_path"])
if not EARLY_WARNING_POLICY_PATH.exists():
    raise FileNotFoundError(f"{EARLY_WARNING_POLICY_PATH} not found.\nFix: re-run Notebook 42.")
with open(EARLY_WARNING_POLICY_PATH, "r", encoding="utf-8") as f:
    EARLY_WARNING_POLICY = json.load(f)

Z_THRESHOLD = EARLY_WARNING_POLICY["z_threshold"]
MIN_STATEMENTS_FOR_BASELINE = EARLY_WARNING_POLICY["min_statements_for_baseline"]
MIN_DEVIATION_COUNT_CANDIDATES = EARLY_WARNING_POLICY["min_deviation_count_candidates"]
CANDIDATE_FEATURES = EARLY_WARNING_POLICY["monitored_features"]["features"]
N_MONITORED_FEATURES = len(CANDIDATE_FEATURES)
EWS_KPI_TARGETS = EARLY_WARNING_POLICY["kpi_targets"]
P6_WINNING_W = NB40_SUMMARY["winning_w"]
P6_RECOMMENDED_FOR_PRODUCTION = NB40_SUMMARY["recommended_for_production"]

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

if "early_warning_models" in PILLAR_DIRS:
    EWS_MODELS_DIR = PILLAR_DIRS["early_warning_models"]
else:
    EWS_MODELS_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "Problem7_Early_Warning_System" / "models"
    )
    print(f"NOTE: 'early_warning_models' not in pillar_dirs -- using fallback: {EWS_MODELS_DIR}")
EWS_MODELS_DIR.mkdir(parents=True, exist_ok=True)

if "early_warning_reports" in PILLAR_DIRS:
    EWS_REPORTS_DIR = PILLAR_DIRS["early_warning_reports"]
else:
    EWS_REPORTS_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "Problem7_Early_Warning_System" / "reports"
    )
    print(f"NOTE: 'early_warning_reports' not in pillar_dirs -- using fallback: {EWS_REPORTS_DIR}")
EWS_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                 : {CONFIG_PATH}")
print(f"Reused Problem 7's real policy from: {EARLY_WARNING_POLICY_PATH}")
print(f"Z_THRESHOLD (ASSUMPTION, reused)              : {Z_THRESHOLD}")
print(f"MIN_STATEMENTS_FOR_BASELINE (ASSUMPTION, reused): {MIN_STATEMENTS_FOR_BASELINE}")
print(f"MIN_DEVIATION_COUNT_CANDIDATES (ASSUMPTION, reused): {MIN_DEVIATION_COUNT_CANDIDATES}")
print(f"Monitored features (reused from Problem 4/6, real): {N_MONITORED_FEATURES}")
print(f"Problem 6 winning window / recommended: W={P6_WINNING_W} / {P6_RECOMMENDED_FOR_PRODUCTION}")
print(f"Champion architecture (Problem 1, measured, reference only -- no training happens in this notebook): "
      f"{CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference)  : {FULL_HISTORY_AUC}")
print(f"Models will be written under : {EWS_MODELS_DIR}")
print(f"Reports will be written under: {EWS_REPORTS_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, log_loss, roc_curve, precision_recall_curve,
        confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses
    (see Notebook 35 Section 3 for the full history of why): the real known
    current nested Phase/Problem path is checked FIRST (never trust
    PILLAR_DIRS alone for a pillar that predates a folder reorg), then
    PILLAR_DIRS, then the legacy root-level path, then whatever a summary
    JSON literally recorded."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + f"\n\nnotebook_02_summary.json['output_files'] keys: "
        f"{sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"train_split.csv (internal train, Notebook 02's real split) : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA RE-VERIFICATION -- MONITORED COLUMNS STILL PRESENT
# =============================================================================
_section("SECTION 4: Live Schema Re-Verification -- Monitored Columns Still Present")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)

_missing_cols = set(CANDIDATE_FEATURES) - _header_cols
if _missing_cols:
    raise RuntimeError(
        f"{len(_missing_cols)} monitored column(s) from Notebook 42's policy are not present in the real "
        f"raw CSV header: {sorted(_missing_cols)}\nFix: investigate before proceeding."
    )
if "S_2" not in _header_cols:
    raise RuntimeError("S_2 (statement date) column not found in the raw CSV -- required to order each "
                        "customer's statements chronologically before computing a rolling baseline.")
print(f"Confirmed all {N_MONITORED_FEATURES} monitored columns from Notebook 42's policy are present "
      "in the live raw CSV header.")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: ROLLING BASELINE / LATEST-STATEMENT FEATURE ENGINEERING --
#            REUSABLE FUNCTION
# =============================================================================
_section("SECTION 5: Rolling Baseline / Latest-Statement Feature Engineering -- Reusable Function")


def build_rolling_zscore_store(csv_path: Path, base_cols: list, min_statements: int) -> "pl.DataFrame":
    """Streams csv_path and returns one aggregated row per customer_ID who has
    >= min_statements real statements (by chronological S_2 date order --
    customers below the threshold are excluded entirely, not silently padded).

    For each monitored base column, computes a BASELINE mean and sample
    standard deviation (ddof=1) from that customer's own statements
    EXCLUDING the most recent one, plus the LATEST statement's raw value --
    exactly the two ingredients Section 7 needs to compute a per-feature
    z-score (latest - baseline_mean) / baseline_std. No model is trained
    here; this is pure per-customer descriptive statistics.

    Reuses the same inf-token-to-null cleaning Notebook 04/39 established
    before any aggregation.
    """
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(pl.col("_n_statements") >= min_statements)
    )

    _is_baseline = pl.col("_row_idx") < (pl.col("_n_statements") - 1)

    agg_exprs = [pl.first("_n_statements").alias("n_statements")]
    for c in base_cols:
        _baseline_val = pl.when(_is_baseline).then(pl.col(c)).otherwise(None)
        agg_exprs += [
            _baseline_val.mean().alias(f"_baseline_mean_{c}"),
            _baseline_val.std(ddof=1).alias(f"_baseline_std_{c}"),
            _baseline_val.count().alias(f"_baseline_n_{c}"),
            pl.col(c).last().alias(f"_latest_{c}"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)
    return grouped.sort("customer_ID").collect(engine="streaming")


print("build_rolling_zscore_store() defined.")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: LOAD LABELS & TRAIN/VALIDATION SPLIT MEMBERSHIP
# =============================================================================
_section("SECTION 6: Load Labels & Train/Validation Split Membership")

labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
print(f"Live-read {RAW_TRAIN_LABELS_PATH.name}: {labels_df.shape[0]:,} labeled customers")

# --- Reuses the EXACT same train/validation split membership Notebook 02
#     established (the same population every prior problem's holdout AUC
#     was measured on), so Problem 7's evaluation is directly comparable to
#     Problems 1/5/6's -- even though, unlike those, NO training happens
#     here (this is an unsupervised, rule-based technique with no
#     train/holdout leakage concern at all). The split is reused purely for
#     apples-to-apples evaluation-population comparability. ---
train_ids_set = set(pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
print(f"Train-split customers (from Notebook 02, reused)     : {len(train_ids_set):,}")
print(f"Validation-split customers (from Notebook 02, reused): {len(val_ids_set):,}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: COMPUTE REAL PER-CUSTOMER ROLLING Z-SCORES & EARLY_WARNING_SCORE
# =============================================================================
_section("SECTION 7: Compute Real Per-Customer Rolling Z-Scores & EARLY_WARNING_SCORE")

print(
    "SCOPE (ASSUMPTION, stated plainly): this notebook computes the rolling z-score baseline/latest "
    "store across ALL customers meeting the baseline-eligibility bar (>= "
    f"{MIN_STATEMENTS_FOR_BASELINE} statements) -- there is no 'training' step to protect from leakage, "
    "since no labels are used anywhere in computing a z-score. The KPI evaluation below is nonetheless "
    "restricted to the SAME validation-split customers Problems 1/5/6 evaluated their holdout AUC on, "
    "purely so this problem's numbers are directly, fairly comparable to theirs -- not because of any "
    "leakage risk specific to this technique."
)

gc.collect()
_t0 = time.time()
zscore_store = build_rolling_zscore_store(RAW_TRAIN_DATA_PATH, CANDIDATE_FEATURES, MIN_STATEMENTS_FOR_BASELINE)
_build_seconds = time.time() - _t0
print(f"Built rolling baseline/latest store for {zscore_store.height:,} eligible customers "
      f"in {_build_seconds:.1f}s. Process RSS: {_rss_gb():.2f} GB")

engineered = zscore_store.join(labels_df, on="customer_ID", how="inner")
del zscore_store
gc.collect()

holdout_df = engineered.filter(pl.col("customer_ID").is_in(val_ids_set))
train_split_eligible_count = engineered.filter(pl.col("customer_ID").is_in(train_ids_set)).height
print(f"Eligible customers with a real label: {engineered.height:,} total -- "
      f"{train_split_eligible_count:,} train-split, {holdout_df.height:,} holdout-split (evaluated below)")
del engineered
gc.collect()

_mean_cols = [f"_baseline_mean_{c}" for c in CANDIDATE_FEATURES]
_std_cols = [f"_baseline_std_{c}" for c in CANDIDATE_FEATURES]
_n_cols = [f"_baseline_n_{c}" for c in CANDIDATE_FEATURES]
_latest_cols = [f"_latest_{c}" for c in CANDIDATE_FEATURES]

mean_arr = holdout_df.select(_mean_cols).to_numpy().astype(np.float64)
std_arr = holdout_df.select(_std_cols).to_numpy().astype(np.float64)
n_arr = holdout_df.select(_n_cols).to_numpy().astype(np.float64)
latest_arr = holdout_df.select(_latest_cols).to_numpy().astype(np.float64)
y_holdout = holdout_df.get_column("target").to_numpy().astype(np.int64)
holdout_customer_ids = holdout_df.get_column("customer_ID").to_numpy()
del holdout_df
gc.collect()

with np.errstate(invalid="ignore", divide="ignore"):
    z_scores = (latest_arr - mean_arr) / std_arr

# --- A per-customer, per-feature z-score is only COMPUTABLE when the
#     baseline has >= 2 non-null points (a real sample std needs >= 2
#     degrees of freedom), the baseline std is strictly positive (a
#     constant baseline has an undefined "how far off is this" answer),
#     and neither the baseline mean nor the latest value is null. Where not
#     computable, this feature honestly does NOT count toward
#     EARLY_WARNING_SCORE for that customer -- it is excluded, not treated
#     as either deviating or not deviating. ---
z_computable_mask = (n_arr >= 2) & (std_arr > 0) & ~np.isnan(latest_arr) & ~np.isnan(mean_arr) & ~np.isnan(std_arr)
z_scores = np.where(z_computable_mask, z_scores, np.nan)

deviates_mask = np.where(z_computable_mask, np.abs(z_scores) >= Z_THRESHOLD, False)
EARLY_WARNING_SCORE = deviates_mask.sum(axis=1).astype(np.int64)
Z_COMPUTABLE_COUNT = z_computable_mask.sum(axis=1).astype(np.int64)

print(f"Holdout customers scored: {len(EARLY_WARNING_SCORE):,}")
print(f"EARLY_WARNING_SCORE distribution (real, measured): "
      f"min={EARLY_WARNING_SCORE.min()}, median={np.median(EARLY_WARNING_SCORE):.1f}, "
      f"mean={EARLY_WARNING_SCORE.mean():.2f}, max={EARLY_WARNING_SCORE.max()}, "
      f"of {N_MONITORED_FEATURES} monitored features")
print(f"Z-computable features per customer (real, measured): "
      f"min={Z_COMPUTABLE_COUNT.min()}, median={np.median(Z_COMPUTABLE_COUNT):.1f}, "
      f"mean={Z_COMPUTABLE_COUNT.mean():.2f}, max={Z_COMPUTABLE_COUNT.max()}")
print(f"Process RSS: {_rss_gb():.2f} GB")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: THRESHOLD-FREE EVALUATION -- ROC-AUC, PR-AUC, LOG LOSS OF THE
#            CONTINUOUS EARLY_WARNING_SCORE (SECONDARY, NON-GATING REQUIREMENT)
# =============================================================================
_section("SECTION 8: Threshold-Free Evaluation -- Continuous EARLY_WARNING_SCORE")

print(
    "Per Notebook 42's KPI Section 8, this is a SECONDARY, non-gating reporting requirement -- reported "
    "for comparability with Problems 1/5/6's own AUC figures, not as a pass/fail bar for this "
    "unsupervised, rule-based technique. A normalized score (EARLY_WARNING_SCORE / monitored feature "
    "count, in [0, 1]) stands in for a probability so ROC-AUC/PR-AUC/log-loss are all well-defined -- "
    "ranking is identical to the raw integer score, log-loss additionally needs values in (0, 1)."
)

BASE_DEFAULT_RATE_HOLDOUT = float(y_holdout.mean())
EARLY_WARNING_SCORE_NORMALIZED = EARLY_WARNING_SCORE.astype(np.float64) / N_MONITORED_FEATURES

holdout_auc = float(roc_auc_score(y_holdout, EARLY_WARNING_SCORE_NORMALIZED))
holdout_pr_auc = float(average_precision_score(y_holdout, EARLY_WARNING_SCORE_NORMALIZED))
_eps = 1e-7
_clipped_score = np.clip(EARLY_WARNING_SCORE_NORMALIZED, _eps, 1.0 - _eps)
holdout_log_loss = float(log_loss(y_holdout, _clipped_score, labels=[0, 1]))

fpr, tpr, _roc_thresholds = roc_curve(y_holdout, EARLY_WARNING_SCORE_NORMALIZED)
pr_precision, pr_recall, _pr_thresholds = precision_recall_curve(y_holdout, EARLY_WARNING_SCORE_NORMALIZED)

auc_retention_pct = float(holdout_auc / FULL_HISTORY_AUC * 100.0) if FULL_HISTORY_AUC else None

print(f"Base holdout default rate (real, measured)        : {BASE_DEFAULT_RATE_HOLDOUT:.4f} "
      f"({BASE_DEFAULT_RATE_HOLDOUT*100:.2f}%)")
print(f"EARLY_WARNING_SCORE (normalized) ROC-AUC vs. real label: {holdout_auc:.4f}")
print(f"EARLY_WARNING_SCORE (normalized) PR-AUC vs. real label : {holdout_pr_auc:.4f}")
print(f"EARLY_WARNING_SCORE (normalized) Log Loss              : {holdout_log_loss:.4f}")
if auc_retention_pct is not None:
    print(f"(Informational only, not a KPI here) Retains {auc_retention_pct:.1f}% of Problem 1's "
          f"full-history champion AUC ({FULL_HISTORY_AUC:.4f}) -- an honestly lower figure is the "
          "EXPECTED outcome for a rule-based control-chart technique vs. a trained 400-tree GBM, not a "
          "failure of this notebook.")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: SWEEP MIN_DEVIATION_COUNT_CANDIDATES -- FULL THRESHOLD METRICS
#            SUITE + REAL DEFAULT-RATE LIFT KPI AT EACH CANDIDATE
# =============================================================================
_section("SECTION 9: Sweep MIN_DEVIATION_COUNT_CANDIDATES -- Full Metrics Suite + Lift KPI")

print(
    f"PRIMARY KPI (per Notebook 42 policy): default-rate lift among ALERTED customers >= "
    f"{EWS_KPI_TARGETS['min_default_rate_lift']}x the base holdout default rate "
    f"({BASE_DEFAULT_RATE_HOLDOUT*100:.2f}%). Evaluated at each candidate ALERT threshold below -- "
    "ALERT = EARLY_WARNING_SCORE >= candidate, treated as the binary prediction for the standing "
    "full-metrics-suite requirement."
)

EWS_MODELING_RESULTS = {}
_n_holdout = len(y_holdout)

print(f"\n{'candidate':>9} {'n_alert':>8} {'%alert':>7} {'default%':>9} {'lift':>6} {'KPI':>8} "
      f"{'acc':>6} {'prec':>6} {'rec':>6} {'f1':>6} {'spec':>6} {'mcc':>6}  confusion(tn,fp,fn,tp)")

for _cand in MIN_DEVIATION_COUNT_CANDIDATES:
    pred = (EARLY_WARNING_SCORE >= _cand).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_holdout, pred, labels=[0, 1]).ravel()
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    n_alerted = int(pred.sum())
    default_rate_alerted = float(y_holdout[pred == 1].mean()) if n_alerted > 0 else None
    lift = (
        (default_rate_alerted / BASE_DEFAULT_RATE_HOLDOUT)
        if (default_rate_alerted is not None and BASE_DEFAULT_RATE_HOLDOUT > 0) else None
    )
    meets_kpi = bool(lift is not None and lift >= EWS_KPI_TARGETS["min_default_rate_lift"])

    metrics = {
        "candidate_min_deviation_count": int(_cand),
        "n_alerted": n_alerted,
        "pct_alerted": 100.0 * n_alerted / _n_holdout,
        "default_rate_alerted": default_rate_alerted,
        "base_default_rate_holdout": BASE_DEFAULT_RATE_HOLDOUT,
        "default_rate_lift": lift,
        "meets_kpi_target": meets_kpi,
        "accuracy": float(accuracy_score(y_holdout, pred)),
        "precision": float(precision_score(y_holdout, pred, zero_division=0)),
        "recall": float(recall_score(y_holdout, pred, zero_division=0)),
        "f1": float(f1_score(y_holdout, pred, zero_division=0)),
        "specificity": specificity,
        "mcc": float(matthews_corrcoef(y_holdout, pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    EWS_MODELING_RESULTS[int(_cand)] = metrics

    _default_pct_str = f"{default_rate_alerted*100:.2f}" if default_rate_alerted is not None else "n/a"
    _lift_str = f"{lift:.2f}x" if lift is not None else "n/a"
    print(f"{_cand:>9} {n_alerted:>8,} {metrics['pct_alerted']:>6.2f}% {_default_pct_str:>8}% {_lift_str:>6} "
          f"{('MET' if meets_kpi else 'NOT MET'):>8} "
          f"{metrics['accuracy']:>6.4f} {metrics['precision']:>6.4f} {metrics['recall']:>6.4f} "
          f"{metrics['f1']:>6.4f} {metrics['specificity']:>6.4f} {metrics['mcc']:>6.4f}  "
          f"({tn:,}, {fp:,}, {fn:,}, {tp:,})")

_candidates_meeting_kpi = [c for c, m in EWS_MODELING_RESULTS.items() if m["meets_kpi_target"]]
_meeting_kpi_str = (
    str(_candidates_meeting_kpi) if _candidates_meeting_kpi
    else "NONE -- reported plainly, see Notebook 44 for the honest winning-candidate selection."
)
print(f"\nCandidate(s) meeting the >= {EWS_KPI_TARGETS['min_default_rate_lift']}x lift KPI: {_meeting_kpi_str}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: ROC + PRECISION-RECALL CURVES & DEFAULT-RATE LIFT CHART (INLINE)
# =============================================================================
_section("SECTION 10: ROC + Precision-Recall Curves & Default-Rate Lift Chart (Inline)")

CHARTS_DIR = EWS_REPORTS_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

fig1, ax1 = plt.subplots(figsize=(7, 6))
ax1.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"EARLY_WARNING_SCORE (AUC = {holdout_auc:.4f})")
ax1.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random (AUC = 0.500)")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("Problem 7: ROC Curve -- Rolling Z-Score Early Warning Score\n(real, holdout-split, "
               "normalized EARLY_WARNING_SCORE)", fontsize=11)
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)
fig1.tight_layout()
chart1_path = CHARTS_DIR / "notebook_43_roc_curve.png"
fig1.savefig(chart1_path, dpi=150)
plt.show()
plt.close(fig1)
print(f"Saved: {chart1_path}")

fig2, ax2 = plt.subplots(figsize=(7, 6))
ax2.plot(pr_recall, pr_precision, color="#d62728", lw=2, label=f"EARLY_WARNING_SCORE (PR-AUC = {holdout_pr_auc:.4f})")
ax2.axhline(BASE_DEFAULT_RATE_HOLDOUT, color="gray", lw=1, linestyle="--",
            label=f"Base rate ({BASE_DEFAULT_RATE_HOLDOUT:.3f})")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Problem 7: Precision-Recall Curve -- Rolling Z-Score Early Warning Score\n(real, holdout-split)",
              fontsize=11)
ax2.legend(loc="upper right")
ax2.grid(alpha=0.3)
fig2.tight_layout()
chart2_path = CHARTS_DIR / "notebook_43_pr_curve.png"
fig2.savefig(chart2_path, dpi=150)
plt.show()
plt.close(fig2)
print(f"Saved: {chart2_path}")

fig3, ax3 = plt.subplots(figsize=(7, 5))
_cands_sorted = sorted(EWS_MODELING_RESULTS.keys())
_lifts = [EWS_MODELING_RESULTS[c]["default_rate_lift"] or 0.0 for c in _cands_sorted]
_colors = ["#2ca02c" if EWS_MODELING_RESULTS[c]["meets_kpi_target"] else "#7f7f7f" for c in _cands_sorted]
_bars = ax3.bar([str(c) for c in _cands_sorted], _lifts, color=_colors)
ax3.axhline(EWS_KPI_TARGETS["min_default_rate_lift"], color="#d62728", lw=1.5, linestyle="--",
            label=f"KPI target ({EWS_KPI_TARGETS['min_default_rate_lift']}x)")
ax3.set_xlabel("MIN_DEVIATION_COUNT candidate")
ax3.set_ylabel("Default-rate lift (alerted vs. base population)")
ax3.set_title("Problem 7: Real Default-Rate Lift by Alert-Threshold Candidate\n(green = meets KPI, gray = does not)",
              fontsize=11)
ax3.legend(loc="best")
ax3.grid(alpha=0.3, axis="y")
fig3.tight_layout()
chart3_path = CHARTS_DIR / "notebook_43_lift_by_candidate.png"
fig3.savefig(chart3_path, dpi=150)
plt.show()
plt.close(fig3)
print(f"Saved: {chart3_path}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: WRITE MODELING RESULTS ARTIFACT
# =============================================================================
_section("SECTION 11: Write Modeling Results Artifact")

MODELING_RESULTS_ARTIFACT = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 7 -- Early Warning System (Rolling Z-Score Trend-Deviation Detection)",
    "z_threshold": Z_THRESHOLD,
    "min_statements_for_baseline": MIN_STATEMENTS_FOR_BASELINE,
    "monitored_feature_count": N_MONITORED_FEATURES,
    "n_holdout_customers_scored": _n_holdout,
    "base_default_rate_holdout": BASE_DEFAULT_RATE_HOLDOUT,
    "early_warning_score_distribution": {
        "min": int(EARLY_WARNING_SCORE.min()),
        "median": float(np.median(EARLY_WARNING_SCORE)),
        "mean": float(EARLY_WARNING_SCORE.mean()),
        "max": int(EARLY_WARNING_SCORE.max()),
    },
    "z_computable_feature_count_distribution": {
        "min": int(Z_COMPUTABLE_COUNT.min()),
        "median": float(np.median(Z_COMPUTABLE_COUNT)),
        "mean": float(Z_COMPUTABLE_COUNT.mean()),
        "max": int(Z_COMPUTABLE_COUNT.max()),
    },
    "secondary_threshold_free_metrics": {
        "roc_auc": holdout_auc,
        "pr_auc": holdout_pr_auc,
        "log_loss": holdout_log_loss,
        "auc_retention_pct_of_full_history": auc_retention_pct,
        "full_history_reference_auc": FULL_HISTORY_AUC,
    },
    "candidate_results": EWS_MODELING_RESULTS,
    "candidates_meeting_kpi": _candidates_meeting_kpi,
    "chart_paths": {
        "roc_curve": str(chart1_path),
        "pr_curve": str(chart2_path),
        "lift_by_candidate": str(chart3_path),
    },
    "random_seed": RANDOM_SEED,
}
modeling_results_path = EWS_MODELS_DIR / "early_warning_modeling_results.json"
with open(modeling_results_path, "w", encoding="utf-8") as f:
    json.dump(MODELING_RESULTS_ARTIFACT, f, indent=2)
print(f"Wrote: {modeling_results_path}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 12: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Modeling results file was written", modeling_results_path.exists())
_all_checks_passed &= _check("Every candidate in the policy has a result",
                              set(EWS_MODELING_RESULTS.keys()) == set(int(c) for c in MIN_DEVIATION_COUNT_CANDIDATES))
_all_checks_passed &= _check("ROC-AUC is a real probability-like value in [0, 1]", 0.0 <= holdout_auc <= 1.0)
_all_checks_passed &= _check("PR-AUC is a real probability-like value in [0, 1]", 0.0 <= holdout_pr_auc <= 1.0)
_all_checks_passed &= _check("Base default rate matches the real holdout label mean",
                              abs(BASE_DEFAULT_RATE_HOLDOUT - float(y_holdout.mean())) < 1e-9)
_all_checks_passed &= _check(
    "pct_alerted is non-increasing as the candidate threshold rises (a stricter bar alerts no more customers)",
    all(EWS_MODELING_RESULTS[_cands_sorted[i]]["pct_alerted"] >= EWS_MODELING_RESULTS[_cands_sorted[i + 1]]["pct_alerted"]
        for i in range(len(_cands_sorted) - 1))
)
_all_checks_passed &= _check(
    "Every candidate's confusion matrix sums to the real holdout customer count",
    all(sum(EWS_MODELING_RESULTS[c]["confusion_matrix"].values()) == _n_holdout for c in EWS_MODELING_RESULTS)
)
_all_checks_passed &= _check("Z-computable feature count never exceeds the monitored feature count",
                              int(Z_COMPUTABLE_COUNT.max()) <= N_MONITORED_FEATURES)
_all_checks_passed &= _check("EARLY_WARNING_SCORE never exceeds the monitored feature count",
                              int(EARLY_WARNING_SCORE.max()) <= N_MONITORED_FEATURES)
_all_checks_passed &= _check("All three chart PNGs were written",
                              chart1_path.exists() and chart2_path.exists() and chart3_path.exists())
_all_checks_passed &= _check("Reused Problem 7's real policy Z_THRESHOLD/candidates (no re-guessing)",
                              MIN_DEVIATION_COUNT_CANDIDATES == EARLY_WARNING_POLICY["min_deviation_count_candidates"])

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 12 complete -- all checks passed.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 43 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 13: Write Notebook 43 Summary Artifact")

NB43_SUMMARY = {
    "notebook": "43_early_warning_system_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "modeling_results_path": str(modeling_results_path),
    "n_holdout_customers_scored": _n_holdout,
    "base_default_rate_holdout": BASE_DEFAULT_RATE_HOLDOUT,
    "secondary_roc_auc": holdout_auc,
    "secondary_pr_auc": holdout_pr_auc,
    "candidates_meeting_kpi": _candidates_meeting_kpi,
    "min_deviation_count_candidates": MIN_DEVIATION_COUNT_CANDIDATES,
    "chart_paths": {
        "roc_curve": str(chart1_path),
        "pr_curve": str(chart2_path),
        "lift_by_candidate": str(chart3_path),
    },
    "random_seed": RANDOM_SEED,
}
NB43_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_43_summary.json"
with open(NB43_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB43_SUMMARY, f, indent=2)
print(f"Wrote: {NB43_SUMMARY_PATH}")

_section("NOTEBOOK 43 COMPLETE")
print(f"Holdout customers scored (real, measured)      : {_n_holdout:,}")
print(f"Base holdout default rate (real, measured)      : {BASE_DEFAULT_RATE_HOLDOUT*100:.2f}%")
print(f"Secondary ROC-AUC / PR-AUC (real, measured)     : {holdout_auc:.4f} / {holdout_pr_auc:.4f}")
print(f"Candidate(s) meeting the >= {EWS_KPI_TARGETS['min_default_rate_lift']}x lift KPI: "
      f"{_candidates_meeting_kpi if _candidates_meeting_kpi else 'NONE'}")
print(f"Modeling results written to: {modeling_results_path}")
print(
    "\nNext: 44_early_warning_system_validation_deployment.ipynb -- selects the winning "
    "MIN_DEVIATION_COUNT candidate (meets the lift KPI, shortest/most-sensitive such candidate; else the "
    "best-performing candidate flagged NOT RECOMMENDED, same honest-selection pattern as Notebooks 36/40), "
    "runs statistical validation, and packages a deployable real-time alert service."
)
